In [6]:
#| default_exp hyperparameter_tuning

## Hyperparameter Tuning

Peshbeen provides unified hyperparameter tuning via Optuna across all model families:
- **`optuna_tune`**: For univariate forecasters (`ml_forecaster`, `ml_direct_forecaster`, `arima`, `ets`, `ms_arr`, `glm`). Supports rolling and expanding time-series cross-validation, lag optimization, Box-Cox variance stabilization, and automated Pareto-based exogenous feature selection.
- **`optuna_tune_multi`**: For multi-series panel models (`panel_forecaster`, `ml_multi_forecaster`). Evaluates cross-validation performance across multiple series simultaneously with options to score on individual or subset target series.


### Hyperparameter tuning for machine learning models using `optuna_tune`


In [7]:
from peshbeen.datasets import load_airline_passengers
from peshbeen.metrics import MAE, RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import optuna_tune

# Load monthly airline passengers dataset
pax = load_airline_passengers()
train = pax[:-12]
test = pax[-12:]

# Create base ML forecaster
ml_model = ml_forecaster(
    model=LGBMRegressor(verbose=-1, random_state=42),
    target_col='passengers',
    lags=12
)
ml_model.fit(train)

# Define hyperparameter search space using Optuna trial suggestions
lgb_param_space = {
    "learning_rate":    lambda t: t.suggest_float("learning_rate", 0.01, 0.3, log=True),
    "num_leaves":       lambda t: t.suggest_int("num_leaves", 10, 50),
    "max_depth":        lambda t: t.suggest_int("max_depth", 2, 8),
    "lags":             lambda t: t.suggest_categorical("lags", [[1, 2, 12], [1, 12], [1, 2, 3, 12]]),
    "box_cox":          lambda t: t.suggest_float("box_cox", 0.0, 2.0),
    "box_cox_biasadj":  lambda t: t.suggest_categorical("box_cox_biasadj", [True, False])
}

# Run cross-validated hyperparameter optimization
best_params, best_lags, other_, _ = optuna_tune(
    model=ml_model,
    df=train,
    cv_split=3,
    step_size=6,
    test_size=12,
    eval_metric=RMSE,
    eval_num=10,
    param_space=lgb_param_space,
    verbose=False
)
print("Best model hyperparameters:", best_params)
print("Best lag structure:", best_lags)
print("Best transformations/box-cox:", other_)


Best model hyperparameters: {'learning_rate': 0.05748924681991978, 'num_leaves': 34, 'max_depth': 2}
Best lag structure: [1, 2, 12]
Best transformations/box-cox: {'box_cox': 1.8977710745066665, 'box_cox_biasadj': True}


In [8]:
# Re-fit a fresh model using the winning parameters and forecast future horizon
final_model = ml_forecaster(
    model=LGBMRegressor(**best_params, verbose=-1, random_state=42),
    target_col='passengers',
    lags=best_lags,
    box_cox=other_.get('box_cox', None),
    box_cox_biasadj=other_.get('box_cox_biasadj', False)
)
final_model.fit(train)
ml_forecasts = final_model.forecast(H=12)
print("Forecasts for next 12 months:", ml_forecasts[:4])


Forecasts for next 12 months: [437.40109234 412.37869871 452.59956269 452.59956269]


### Selecting the best ETS (Error, Trend, Seasonality) model using `optuna_tune`


In [11]:
from peshbeen.models import ets

ets_param_space = {
    "trend":              lambda t: t.suggest_categorical("trend", ["add", "mul", None]),
    "seasonal":           lambda t: t.suggest_categorical("seasonal", ["add", "mul", None]),
    "seasonal_periods":   lambda t: 12,
    "smoothing_level":    lambda t: t.suggest_float("smoothing_level", 0.01, 0.99),
    "smoothing_trend":    lambda t: t.suggest_float("smoothing_trend", 0.01, 0.99),
    "smoothing_seasonal": lambda t: t.suggest_float("smoothing_seasonal", 0.01, 0.99)
}

ets_model = ets(target_col='passengers')
best_params, _, other_, _ = optuna_tune(
    model=ets_model,
    df=train,
    cv_split=3,
    step_size=6,
    test_size=12,
    eval_metric=RMSE,
    eval_num=10,
    param_space=ets_param_space,
    verbose=False
)
print("Best ETS parameters:", best_params)
## add seasonal_periods to the best parameters
best_params["seasonal_periods"] = 12

Best ETS parameters: {'trend': 'add', 'seasonal': 'mul', 'smoothing_level': 0.6216018566289179, 'smoothing_trend': 0.044204810055873955, 'smoothing_seasonal': 0.9825154121511303}


In [12]:
# Forecast with the best ETS parameters
best_params.update(other_)
tuned_ets = ets(target_col='passengers', **best_params)
tuned_ets.fit(train)
ets_forecasts = tuned_ets.forecast(H=12)
print("ETS forecasts:", ets_forecasts[:4])


ETS forecasts: [431.04450457 400.13769768 474.33078426 455.3642518 ]


In [13]:
# Evaluate cross-validation performance with best parameters
cv_df = tuned_ets.cross_validate(
    df=train,
    cv_split=3,
    step_size=6,
    test_size=12,
    metrics=[RMSE, MAE]
)
cv_df.head()


,cutoff,fold_index,horizon,split,y_true,y_pred
0,1957-12-01,1958-01-01,1,fold_1,340,357.402752
1,1957-12-01,1958-02-01,2,fold_1,318,338.442605
2,1957-12-01,1958-03-01,3,fold_1,362,408.145048
3,1957-12-01,1958-04-01,4,fold_1,348,389.666514
4,1957-12-01,1958-05-01,5,fold_1,363,387.606278


### Selecting the best orders for an ARIMA model using `optuna_tune`

> **Tip:** For automated ARIMA model selection, Peshbeen also provides `auto_arima` natively (implementing the Hyndman-Khandakar stepwise algorithm and repeated KPSS stationarity tests) with `.top_models()` and `.get_orders()`!

In [15]:
from peshbeen.models import arima

# Search space for SARIMA orders on monthly airline passenger series
candidate_orders = [(1, 1, 1), (0, 1, 1), (1, 1, 0), (2, 1, 1)]
candidate_seasonal = [(1, 1, 1), (0, 1, 1), (1, 1, 0), (0, 1, 0)]

arima_param_space = {
    "order":            lambda t: t.suggest_categorical("order", candidate_orders),
    "seasonal_order":   lambda t: t.suggest_categorical("seasonal_order", candidate_seasonal),
    "seasonal_length":  lambda t: 12,
    "box_cox":          lambda t: t.suggest_float("box_cox", 0.0, 2.0)
}

arima_model = arima(target_col='passengers')
best_params, _, other_, _ = optuna_tune(
    model=arima_model,
    df=train,
    cv_split=3,
    step_size=6,
    test_size=12,
    eval_metric=RMSE,
    eval_num=6,
    param_space=arima_param_space,
    verbose=False
)
print("Best ARIMA orders:", best_params)
best_params["seasonal_periods"] = 12


Best ARIMA orders: {'order': (2, 1, 1), 'seasonal_order': (0, 1, 0)}


### Forward and backward feature selection

Hyperparameter tuning searches over model settings; **feature selection** searches over *which lags, exogenous columns, and lag-transforms* to feed the model. peshbeen provides `forward_feature_selection` (start empty, greedily add the candidate that most improves cross-validation) and `backward_feature_selection` (start from the full set, greedily drop). Both take a *configured but unfitted* `ml_forecaster`, the forecast horizon `H`, a list of `metrics` (selection is driven by the first), and candidate pools via `lags_to_consider`, `candidate_features`, and `transformations`. Each returns a dict with `best_lags`, `best_exogs`, and `best_transforms`.

In [16]:
from peshbeen.model_selection import forward_feature_selection, backward_feature_selection
from peshbeen.metrics import MAE, RMSE
from sklearn.linear_model import LinearRegression

# Configured-but-unfitted template forecaster
fs_model = ml_forecaster(
    model=LinearRegression(),
    target_col='passengers',
    lags=12,
    difference=1
)

# Forward feature selection over candidate lags 1..12
fwd = forward_feature_selection(
    model=fs_model,
    df=train,
    cv_split=3,
    H=12,
    metric=MAE,
    lags_to_consider=12,
    verbose=False
)
print("Forward-selected features:", fwd)


Forward-selected features: {'best_lags': [], 'best_exogs': [], 'best_transforms': []}


### Hyperparameter tuning for multi-series panel models using `optuna_tune_multi`

When optimizing pooled global models (`panel_forecaster` or `ml_multi_forecaster`), `optuna_tune_multi` evaluates time series cross-validation across all series simultaneously.

Key parameters:
- `target_series`: Specify a subset of series IDs to score, which is helpful when tuning models on a representative subset of critical series.
- `transform_back`: When `False`, scores on transformed scales rather than original units.
- `step_size`: Sliding window step size for fold progression.


In [17]:
from peshbeen.datasets import load_sales
from peshbeen.models import panel_forecaster
from peshbeen.model_selection import optuna_tune_multi
from peshbeen.metrics import RMSE
from lightgbm import LGBMRegressor

# Load multi-series sales and filter to a small subset for demonstration
sales_data = load_sales()
subset_items = ['store_01_item_01', 'store_01_item_02']
df_panel = sales_data[sales_data['store_item'].isin(subset_items)].groupby('store_item').tail(120)

# Base panel model
panel_model = panel_forecaster(
    model=LGBMRegressor(verbose=-1, random_state=42),
    id_col='store_item',
    target_col='sales',
    lags=7
)

# Search space for panel model hyperparameters and pooled lag length
panel_space = {
    "learning_rate": lambda t: t.suggest_float("learning_rate", 0.02, 0.2),
    "num_leaves":    lambda t: t.suggest_int("num_leaves", 10, 40),
    "lags":          lambda t: t.suggest_categorical("lags", [[1, 2, 7], [1, 7], [1, 2, 3, 7]])
}

best_panel_h, best_panel_lags, other_panel = optuna_tune_multi(
    model=panel_model,
    df=df_panel,
    cv_split=3,
    test_size=14,
    eval_metric=RMSE,
    param_space=panel_space,
    eval_num=5,
    verbose=False
)
print("Best panel parameters:", best_panel_h)
print("Best panel lags:", best_panel_lags)


Best panel parameters: {'learning_rate': 0.023705208973244442, 'num_leaves': 40}
Best panel lags: [1, 2, 7]


### The metrics module

All tuning and selection functions accept any metric from `peshbeen.metrics`. Most are **two-argument** functions `metric(y_true, y_pred)`:

- `MAPE`, `SMAPE`, `WMAPE` — percentage errors
- `MAE`, `MSE`, `RMSE` — absolute / squared errors
- `CFE`, `CFE_ABS` — cumulative forecast error (bias) and its absolute value

The **scaled** metrics are *three-argument* — `metric(y_true, y_pred, y_train)` — because they normalise the error by the in-sample scale of the training series:

- `MASE` — mean absolute scaled error
- `RMSSE` — root mean squared scaled error
- `SRMSE` — scaled RMSE

When you call a scaled metric directly you must pass the training target as the third argument; for example `MASE(y_true, y_pred, y_train=train['admissions'])`.